In [78]:
from google.cloud import storage
import os
from dotenv import load_dotenv
import pandas as pd
from io import BytesIO

load_dotenv()

True

In [79]:
GCS_BUCKET = os.getenv("GCS_BUCKET", "").strip()
GCS_MEDAL_FETCH = os.getenv("GCS_MEDAL_FETCH", "").strip()
SEASON = int(os.getenv("SEASON", ""))
GCS_BLOB_FETCH = f"{GCS_MEDAL_FETCH}/season={SEASON}/driver.parquet"

In [80]:
DRIVER_NUMBER_TO_COUNTRY = {
    1: "United Kingdom",    
    3: "Netherlands",     
    5: "Brazil",          
    6: "France",          
    10: "France",         
    11: "Mexico",         
    12: "Italy",          
    14: "Spain",          
    16: "Monaco",         
    18: "Canada",         
    23: "Thailand",       
    27: "Germany",        
    30: "New Zealand",    
    31: "France",         
    41: "United Kingdom", 
    43: "Argentina",      
    44: "United Kingdom", 
    55: "Spain",           
    63: "United Kingdom", 
    77: "Finland",        
    81: "Australia",       
    87: "United Kingdom", 
}

In [81]:
client = storage.Client()
bucket = client.bucket(GCS_BUCKET)
blob = bucket.blob(GCS_BLOB_FETCH)

df = pd.read_parquet(BytesIO(blob.download_as_bytes()))

In [82]:
columns_to_drop = ["meeting_key", "session_key", "country_code"]
df = df.drop(columns=columns_to_drop)

In [83]:
df.isna().sum()

driver_number     0
broadcast_name    0
full_name         0
name_acronym      0
team_name         0
team_colour       0
first_name        0
last_name         0
headshot_url      0
dtype: int64

In [84]:
df.duplicated(subset="driver_number").sum()

np.int64(0)

In [85]:
def driver_number_to_country(n):
    key = int(n)
    return DRIVER_NUMBER_TO_COUNTRY.get(key, "Unknown")

**DRIVER NUMBER**

In [86]:
df["driver_number"].value_counts()

driver_number
1     1
3     1
5     1
6     1
10    1
11    1
12    1
14    1
16    1
18    1
23    1
27    1
30    1
31    1
41    1
43    1
44    1
55    1
63    1
77    1
81    1
87    1
Name: count, dtype: int64

In [87]:
df["country"] = df["driver_number"].map(driver_number_to_country)

In [88]:
columns_to_strip = ["broadcast_name", "full_name", "name_acronym", "team_name", "team_colour", "first_name", "last_name", "headshot_url", "country"]
for col in columns_to_strip:
    df[col] = df[col].str.strip()

In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   driver_number   22 non-null     int64 
 1   broadcast_name  22 non-null     object
 2   full_name       22 non-null     object
 3   name_acronym    22 non-null     object
 4   team_name       22 non-null     object
 5   team_colour     22 non-null     object
 6   first_name      22 non-null     object
 7   last_name       22 non-null     object
 8   headshot_url    22 non-null     object
 9   country         22 non-null     object
dtypes: int64(1), object(9)
memory usage: 1.8+ KB


In [90]:
df.head()

,driver_number,broadcast_name,full_name,name_acronym,team_name,team_colour,first_name,last_name,headshot_url,country
0,1,L NORRIS,Lando NORRIS,NOR,McLaren,F47600,Lando,Norris,https://media.formula1.com/d_driver_fallback_i...,United Kingdom
1,3,M VERSTAPPEN,Max VERSTAPPEN,VER,Red Bull Racing,4781D7,Max,Verstappen,https://media.formula1.com/d_driver_fallback_i...,Netherlands
2,5,G BORTOLETO,Gabriel BORTOLETO,BOR,Audi,F50537,Gabriel,Bortoleto,https://media.formula1.com/d_driver_fallback_i...,Brazil
3,6,I HADJAR,Isack HADJAR,HAD,Red Bull Racing,4781D7,Isack,Hadjar,https://media.formula1.com/d_driver_fallback_i...,France
4,10,P GASLY,Pierre GASLY,GAS,Alpine,00A1E8,Pierre,Gasly,https://media.formula1.com/d_driver_fallback_i...,France
